### Homework 06: Concurrency

## Due Date: Apr 10, 2026, 11:00pm

#### Firstname Lastname: Dayne Lee

#### E-mail: dl5635@nyu.edu

#### Enter your solutions and submit this notebook


---

**Problem 1** **(60 Points)**

Let us consider the Gamma function, or the Euler integral of the second kind: 

$$\Gamma(x) = \int_{0} ^ \infty t ^{x - 1} e^{-t} dt, $$

and in this HW we consider real $x > 0$.

(Here is more on the Gamma function https://en.wikipedia.org/wiki/Gamma_function .
It is not needed for this HW assignment.) 

**1.1 (Points 15)**: 

Write a function (in the cell below) that sequentially calculates the given Gamma integral.


In [1]:
import math

def calculate_gamma(x, bound_1, bound_2, number_of_steps):
    dt = (bound_2 - bound_1) / number_of_steps
    
    total = 0.0
    t = bound_1 + dt
    
    for _ in range(number_of_steps):
        val = (t ** (x - 1)) * math.exp(-t)
        total += val * dt
        t += dt
    
    return total

**1.2 (Points 5)** 

Evaluate, $\Gamma(6)$ by using `calculate_gamma(x, bound_1, bound_2, number_of_steps)` and the error of this computation.


As arguments, use `x=6, bound_1=0, bound_2=1000, number_of_steps=10_000_000`. We know that $\Gamma(x) = x!$, so $\Gamma(6) = 5! = 120$. 


In [2]:
x = 6
bound_1 = 0
bound_2 = 1000
number_of_steps = 10_000_000

approx = calculate_gamma(x, bound_1, bound_2, number_of_steps)
true_val = math.factorial(5)  # 120

error = abs(approx - true_val)

print("Approximation:", approx)
print("True value:", true_val)
print("Error:", error)

Approximation: 120.00000000011516
True value: 120
Error: 1.1516476661199704e-10


---

Write two functions to calculate $\Gamma(x)$ by using:



**1.3.1 (Points 15)**
**threading** with N=4 threads; 

**1.3.2 (Points 15)**
**multiprocessing** with N=4 processes. 


**1.3.3 (Points 10)** 
Compare the times of the three versions and write a short explanation of what you are observing.

How does the answer change when N=8 and why?

    

In [3]:
import time

from gamma import gamma_threading, gamma_multiprocessing

x = 6
b1, b2 = 0, 1000
steps = 10_000_000


# sequential
t0 = time.time()
seq = calculate_gamma(x, b1, b2, steps)
t1 = time.time()

# threading
t2 = time.time()
thr = gamma_threading(x, b1, b2, steps, N=4)
t3 = time.time()

# multiprocessing
t4 = time.time()
mp = gamma_multiprocessing(x, b1, b2, steps, N=4)
t5 = time.time()

print("Sequential:", t1 - t0)
print("Threading :", t3 - t2)
print("Multiproc :", t5 - t4)

Sequential: 1.360563039779663
Threading : 1.321727991104126
Multiproc : 0.39547014236450195


#### 1.3.3

**Comparing the times of the three versions**: Sequential is the slowest, followed by multithreading with a minor difference (almost the same), and multiprocessing is the fastest. Because of the GIL in a CPU-bound task, multithreading is not effective, whereas multiprocessing is 4x faster than the sequential version since it truly uses multiple cores.

**How does the answer change when N=8 and why?**: There would be no difference in a multithreading version. For multiprocessing, assuming our machine has more than 8 cores, it may be 8x faster than the sequential version.

---

**Problem 2 (40 points)**

__Website uptime__ is the time that a website or web service is available to the users over a given period.

The task is to build an application that checks the uptime of websites. 

- The application should go over a list of website URLs and checks if those websites are up.
- Instead of performing a classic HTTP GET request, it performs a HEAD request so that it does not affect traffic significantly.
- If the HTTP status is in the danger ranges (400+, 500+), a message is casted. 

Here are some useful functions:

In [4]:
#### _website uptimer_ ####

import time
import logging
import requests
 
class WebsiteDownException(Exception):
    pass
 
def ping_website(address, timeout=20):
    """
    Check if a website is down. A website is considered down 
    if either the status_code >= 400 or if the timeout expires
     
    Throw a WebsiteDownException if any of the website down conditions are met
    """
    try:
        response = requests.head(address, timeout=timeout)
        if response.status_code >= 400:
            logging.warning("Website %s returned status_code=%s" % (address, response.status_code))
            raise WebsiteDownException()
    except requests.exceptions.RequestException:
        logging.warning("Timeout expired for website %s" % address)
        raise WebsiteDownException()
         
def check_website(address):
    """
    Utility function: check if a website is down, if so, notify the user
    """
    try:
        ping_website(address)
    except WebsiteDownException:
        print('The websie ' + address + ' is down')

---

You need a website list to try our system out. Create your own list or use the following one. 

---

In [5]:
WEBSITE_LIST = [
    'http://amazon.co.uk',
    'http://amazon.com',
    'http://facebook.com',
    'http://google.com',
    'http://google.fr',
    'http://google.es',
    'http://google.co.uk',
    'http://gmail.com',
    'http://stackoverflow.com',
    'http://github.com',
    'http://heroku.com',
    'http://really-cool-available-domain.com',
    'http://djangoproject.com',
    'http://rubyonrails.org',
    'http://basecamp.com',
    'http://trello.com',
    'http://shopify.com',
    'http://another-really-interesting-domain.co',
    'http://airbnb.com',
    'http://instagram.com',
    'http://snapchat.com',
    'http://youtube.com',
    'http://baidu.com',
    'http://yahoo.com',
    'http://live.com',
    'http://linkedin.com',
    'http://netflix.com',
    'http://wordpress.com',
    'http://bing.com',
]

---

A serial version of the _website uptimer_ can be written as: 

---


In [6]:
import time
 
start_time = time.time()
 
for address in WEBSITE_LIST:
    check_website(address)
         
end_time = time.time()        
 
print("Time for Serial: %ssecs" % (end_time - start_time))

The websie http://really-cool-available-domain.com is down
The websie http://another-really-interesting-domain.co is down


The websie http://yahoo.com is down
Time for Serial: 2.5019872188568115secs


You should build two versions of the **website uptimer**, by using:

**2.1 (Points 15)**
**threading** with N=4 threads; 

**2.2 (Points 15)**
**multiprocessing** with N=4 processes. 


**2.3 (Points 10)** 

Compare the times of the three versions and write a short explanation of what you are observing.

How does the answer change when N=8 and why?


In [11]:
import multiprocessing as mp

from uptimer import run_threading, run_multiprocessing

mp.set_start_method("fork", force=True)

# Serial
t0 = time.time()
for addr in WEBSITE_LIST:
    check_website(addr)
t1 = time.time()

print("Serial:", t1 - t0)


# multitheading
t2 = time.time()
run_threading(WEBSITE_LIST, N=4)
t3 = time.time()

print("Multithreading (N=4):", t3 - t2)


# multiprocessing
t4 = time.time()
run_multiprocessing(WEBSITE_LIST, N=4)
t5 = time.time()

print("Multiprocessing (N=4):", t5 - t4)

The websie http://really-cool-available-domain.com is down
The websie http://another-really-interesting-domain.co is down


Serial: 2.2329819202423096
The website http://another-really-interesting-domain.co is down
The website http://really-cool-available-domain.com is down
Multithreading (N=4): 1.0246739387512207


The website http://really-cool-available-domain.com is down


The website http://another-really-interesting-domain.co is down
Multiprocessing (N=4): 0.8678281307220459


#### 2.3

**Comparing the times of the three versions**: Threading and multiprocessing both improve performance over the serial version. In this case, multiprocessing was slightly faster than threading, likely due to true parallel execution without GIL contention. However, since the task is I/O-bound, the difference is small and can vary depending on network conditions and system overhead.

**How does the answer change when N=8 and why?**: Increasing N from 4 to 8 provides only limited improvement because the bottleneck is network latency. Additional workers introduce overhead and resource contention, so the speedup quickly diminishes.